# Vaccine Allocation Optimization with Gurobi

## Question Statement:
A hospital has received a batch of 5000 COVID-19 vaccines. The hospital has 4 vaccine stations - Station A, Station B, Station C, and Station D. Each station has a priority based on the
population it serves (e.g., elderly, children, chronically ill, disabled). The higher the priority
value, the more vulnerable the population it serves. Additionally, each station has a storage
capacity, and it is mandatory that every station receives at least 500 vaccines.

| Station | Priority | Storage Capacity (Vaccines) |
|:-------:|:--------:|:--------------------------:|
|    A    |   0.5    |           2300             |
|    B    |   0.7    |           1700             |
|    C    |   1.0    |           1200             |
|    D    |   0.8    |            900             |

*Table 1: Station priorities and storage capacities.*

Please formulate this problem as a linear program and find the optimal vaccine allocation strategy for this hospital using Gurobi.

### Install Gurobi

In [4]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 100.5 MB/s eta 0:00:00


## Importing Required Libraries

We need Gurobi's Python bindings (`gurobipy`) for optimization, and Google's Colab `userdata` module to securely load our Gurobi license.

* `Model`: used to create new optimization models
* `GRB`: Contains Gurobi constants and enumerations, such as `GRB.MAXIMIZE`, `GRB.CONTINUOUS`, `GRB.INTEGER`, etc.
* `quicksum`: a faster version of the `sum` function optimized for Gurobi
* `Env`: only needed if you are using Colab instead of coding locally


In [1]:
from gurobipy import Model, GRB, quicksum, Env

## Setting Up the Gurobi Environment for Colab
The following block is only needed if you are running Gurobi from Colab

Gurobi cloud usage requires Web License Service (WLS) credentials. Here, we load them from secure Colab storage and create a Gurobi environment.

In [6]:
from google.colab import userdata

# Create an environment with your WLS license
params = {
"REDACTED": "REDACTED",
"REDACTED": "REDACTED",
"REDACTED": REDACTED
}

env = Env(params=params)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value REDACTED
Academic license REDACTED - for non-commercial use only - registered to REDACTED


## Vaccine Distribution Model and Data

* `m = Model("vaccine_distribution", env=env)` create a new Gurobi model, which will be the container for all our variables, constraints, and the optimization process. We name the model "vaccine_distribution" and pass in the Gurobi Env created earlier for licensing.

* Then we create lists for the stations, their vaccine storage capacities, their priority weights, and the total vaccines available.

| Station | Priority | Storage Capacity (Vaccines) |
|:-------:|:--------:|:--------------------------:|
|    A    |   0.5    |           2300             |
|    B    |   0.7    |           1700             |
|    C    |   1.0    |           1200             |
|    D    |   0.8    |            900             |


In [3]:
# Initialize the model
m = Model("vaccine_distribution")

# Define the properties of each station
# We use lists for this purpose
stations = ['A', 'B', 'C', 'D']
capacities = [2300, 1700, 1200, 900]
priorities = [0.5, 0.7, 1, 0.8]
total_vaccines = 5000

Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-08


In [4]:
m # defaults as a continuous instance

<gurobi.Model Continuous instance vaccine_distribution: 0 constrs, 0 vars, Parameter changes: Username=(user-defined)>

### Why does it say `Continuous instance`?
When you create a new Gurobi model with:
```
m = Model("vaccine_distribution", env=env)
```
the model is empty—it has:
* 0 constraints
* 0 variables
* and no variable types set yet.

By default, Gurobi models are called “Continuous instance” when there are no variables or only continuous variables, because “continuous” is the default variable type in Gurobi.


### When does this change?
As soon as you add variables with specific types (e.g., `GRB.INTEGER`, `GRB.BINARY`), the model will change its type.

## Defining Decision Variables
In the following code, we
* Use `m.addVar(...)` to add decision variables to the model `m`
* Create integer variables using `GRB.INTEGER` for the number of vaccines each station receives.
* These variables represent the unknowns the solver will choose.

In [5]:
# Create variables for the number of vaccines to distribute to each station
x = [m.addVar(vtype=GRB.CONTINUOUS, name=f"x_{station}") for station in stations]

In [6]:
x # What does this mean?

[<gurobi.Var *Awaiting Model Update*>,
 <gurobi.Var *Awaiting Model Update*>,
 <gurobi.Var *Awaiting Model Update*>,
 <gurobi.Var *Awaiting Model Update*>]

In [7]:
m # Why is it still Continuous instance?

<gurobi.Model Continuous instance vaccine_distribution: 0 constrs, 0 vars, Parameter changes: Username=(user-defined)>

### Why does this happen?

Gurobi is “lazy” and defers the actual creation of variables in its internal model structure until it needs to, for efficiency.

Once you add all variables and constraints, the model is automatically updated the first time you call:
* `m.update()` or
* `m.optimize()`

In [8]:
m.update()
print(x)
print(m) # “MIP” stands for “Mixed Integer Programming.”

[<gurobi.Var x_A>, <gurobi.Var x_B>, <gurobi.Var x_C>, <gurobi.Var x_D>]
<gurobi.Model Continuous instance vaccine_distribution: 0 constrs, 4 vars, Parameter changes: Username=(user-defined)>


## Objective Function

Our goal is to maximize the "priority-weighted" number of vaccines assigned across all stations. That is, more vulnerable stations (with higher priority) are favored.

$$
\text{Maximize} \quad \sum_{i \in \{A,B,C,D\}} P_i \cdot X_i
$$

In [9]:
# Set the objective to maximize priority-weighted distribution
m.setObjective(quicksum(x[i] * priorities[i] for i in range(len(stations))), GRB.MAXIMIZE)

In [10]:
print(m.getObjective())

0.0


In [11]:
m.update()
print(m.getObjective())
print(m.ModelSense) # ModelSense is either GRB.MAXIMIZE or GRB.MINIMIZE

0.5 x_A + 0.7 x_B + x_C + 0.8 x_D
-1


In [16]:
print("Maximize: ", GRB.MAXIMIZE, "\nMinimize: ", GRB.MINIMIZE)

Maximize:  -1 
Minimize:  1


## Adding Constraints

**a.** No station can exceed its own capacity.
**b.** Each station must receive **at least 500 vaccines**.<br>
**c.** All vaccines must be distributed: the sum must be exactly 5000.<br>

$$
\begin{aligned}
    & &&X_i &&\leq \quad S_i,  && \forall i \in \{A,B,C,D\} \\
    & &&X_i &&\geq \quad 500,  && \forall i \in \{A,B,C,D\} \\
    & \sum_{i \in \{A,B,C,D\}} &&X_i &&= \quad 5000 \\
\end{aligned}
$$

In [12]:
# Add constraint to ensure the total number of vaccines distributed equals the total available
m.addConstr(quicksum(x[i] for i in range(len(stations))) == total_vaccines, "total_vaccines")

# Add constraints to ensure each station receives at least 500 vaccines and does not exceed its capacity
for i in range(len(stations)):
    m.addConstr(x[i] >= 500, f"min_vaccines_{stations[i]}")
    m.addConstr(x[i] <= capacities[i], f"capacity_{stations[i]}")

## Solving the Model

We'll now solve the model and print the allocation for each station.

### Optimization Status Code:
Once an optimize call has returned, the Gurobi Optimizer sets the Status attribute of the model to one of several possible values.
* `GRB.OPTIMAL` means that model was solved to optimality (subject to tolerances), and an optimal solution is available.
* For a complete list of status codes, see: https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html#secstatuscodes

In [13]:
# Solve the model
m.optimize()

# Print the solution
if m.status == GRB.OPTIMAL: # if there is an optimal solution, do the following
    for i in range(len(stations)):
        print(f"Station {stations[i]} receives {x[i].x} vaccines") # x[i].x represents the final value of the decision variable
else:
    print("No optimal solution found")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 9 rows, 4 columns and 12 nonzeros
Model fingerprint: 0xbe6f3d26
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-01, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+02, 5e+03]
Presolve removed 9 rows and 4 columns
Presolve time: 0.04s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.7100000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.06 seconds (0.00 work units)
Optimal objective  3.710000000e+03
Station A receives 1200.0 vaccines
Station B receives 1700.0 vaccines
Station C receives 1200.0 vaccines
Station D receives 900.0 vaccines


In [14]:
x[0]

<gurobi.Var x_A (value 1200.0)>

In [15]:
x[0].x

1200.0

## Save/Load a Model
You might save and load a model for several reasons:

* **Long-running builds:** Constructing large models might take time, so you can build once, save, and later load without reconstruction.
* **Sharing:** Transfer the model between team members or machines without source code.
* **Debugging:** Save the state right before or after modifications to diagnose issues.
* **Scalability:** Try variants of a base model by loading, adjusting, and re-solving without rebuilding from scratch.

### File Format
* **MPS**: This is the most widely used format for storing math programming models. It is recommended to use this format (or REW) for optimization.
* **LP**: This format is a human readable version of MPS file format. It, however, does not preserve column order when read, and typically does not preserve the exact numerical values of the coefficients.
* For more details, visit: https://docs.gurobi.com/projects/optimizer/en/current/reference/fileformats.html

In [16]:
m.write("vaccine.mps")

In [20]:
import gurobipy as gp
loaded_model = gp.read("vaccine.mps") # this load the saved model

Read MPS format model from file vaccine.mps
Reading time = 0.00 seconds
vaccine_distribution: 9 rows, 4 columns, 12 nonzeros


In [21]:
m.write("vaccine.lp") # this is a more human readable format
with open("vaccine.lp", "r") as f:
    print(f.read())

\ Model vaccine_distribution
\ LP format - for model browsing. Use MPS format to capture full model detail.
Maximize
  0.5 x_A + 0.7 x_B + x_C + 0.8 x_D
Subject To
 total_vaccines: x_A + x_B + x_C + x_D = 5000
 min_vaccines_A: x_A >= 500
 capacity_A: x_A <= 2300
 min_vaccines_B: x_B >= 500
 capacity_B: x_B <= 1700
 min_vaccines_C: x_C >= 500
 capacity_C: x_C <= 1200
 min_vaccines_D: x_D >= 500
 capacity_D: x_D <= 900
Bounds
End



## Cleaning Up: Releasing Gurobi Resources

When using Gurobi in cloud environments like Google Colab (with the Web License Service), it's important to explicitly release the model and environment resources when you're done. This ensures that your Gurobi license is made available for others and prevents your container from holding onto it unnecessarily.

You can do this by calling the `dispose()` method on your model and environment objects:

In [ ]:
# Release Gurobi resources and license seat
m.dispose()
env.dispose()

> **Tip:**  
> Failing to call `dispose()` (especially in Colab) may result in unused license seats being tied up, causing issues for you when you want to use the same license in another colab session. Always clean up at the end of your session!

## Summary

This notebook demonstrates how to use Gurobi in Google Colab to solve a simple vaccine allocation optimization problem. We defined decision variables, set an objective function to maximize priority-weighted distribution, and added constraints for total vaccine distribution, minimum allocation per station, and station capacity. The model was solved to find an optimal allocation, and Gurobi resources were properly released.